## Completed and Tested Working good

In [4]:
import sys
sys.path.append('/home/magesh/TrandingProjects/Project/backend/dolphin/TradingStradegy')

In [9]:
import pandas as pd
import numpy as np
from datetime import timedelta,datetime
class IINWMARROWSSignalPredictor:
    def __init__(self, df):
        """
        Initialize the MovAvg class with a DataFrame.
        :param df: pandas DataFrame with columns 'close', 'open', 'high', and 'low'.
        """
        self.df = df

    def calculate_ma(self, series, period, mode):
        """
        Calculate the moving average based on the specified mode.
        :param series: pandas Series to calculate the moving average on.
        :param period: int, the window period for the moving average.
        :param mode: int, the mode of moving average (0: SMA, 1: EMA, 2: SMMA, 3: LWMA).
        :return: pandas Series with the moving average.
        """
        if mode == 0:  # Simple Moving Average (SMA)
            return series.rolling(window=period).mean()
        elif mode == 1:  # Exponential Moving Average (EMA)
            return series.ewm(span=period, adjust=False).mean()
        elif mode == 2:  # Simple Modified Moving Average (SMMA)
            return series.ewm(alpha=1.0 / period).mean()
        elif mode == 3:  # Linear Weighted Moving Average (LWMA)
            weights = np.arange(1, period + 1)
            return series.rolling(period).apply(lambda prices: np.dot(prices, weights) / weights.sum(), raw=True)
        else:
            raise ValueError("Invalid mode for moving average")

    def mainloop(self):
        """
        Main loop to calculate moving averages and detect cross signals.
        :return: pandas DataFrame with 'CrossDown', 'CrossUp', 'FasterMA', and 'SlowerMA' columns.
        """
        # Parameters
        faster_mode = 3  # LWMA
        faster_ma_period = 3
        slower_mode = 3  # LWMA
        slower_ma_period = 3

        # Calculate moving averages
        self.df['FasterMA'] = self.calculate_ma(self.df['close'], faster_ma_period, faster_mode)
        self.df['SlowerMA'] = self.calculate_ma(self.df['open'], slower_ma_period, slower_mode)

        # Calculate range and average range
        self.df['Range'] = self.df['high'] - self.df['low']
        self.df['AvgRange'] = self.df['Range'].rolling(window=10).mean()

        # Initialize columns for cross signals
        self.df['CrossUp'] = np.nan
        self.df['CrossDown'] = np.nan

        # Detect cross signals
        crosses_up = (
            (self.df['FasterMA'] > self.df['SlowerMA']) &
            (self.df['FasterMA'].shift(1) < self.df['SlowerMA'].shift(1)) &
            (self.df['FasterMA'] > self.df['FasterMA'].shift(1))
        )
        crosses_down = (
            (self.df['FasterMA'] < self.df['SlowerMA']) &
            (self.df['FasterMA'].shift(1) > self.df['SlowerMA'].shift(1)) &
            (self.df['FasterMA'] < self.df['FasterMA'].shift(1))
        )

        self.df.loc[crosses_up, 'CrossUp'] = self.df['low'] - self.df['AvgRange'] * 0.3
        self.df.loc[crosses_down, 'CrossDown'] = self.df['high'] + self.df['AvgRange'] * 0.3

        return self.df



In [10]:
dataframe = pd.read_csv('/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/EURJPY_5_Min_testing_new.csv').iloc[::-1]
ba = IINWMARROWSSignalPredictor(dataframe.reset_index(drop=True))
final_df = ba.mainloop()
final_df['UTC'] = pd.to_datetime(final_df['datetime']) + timedelta(hours=5)
final_df['GMT'] = final_df['UTC'] + timedelta(hours=2)
output_file = '/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/outputimmarrowsEurusd.csv'
final_df.to_csv(output_file, index=False)

In [12]:
import pandas as pd
import numpy as np
import ta
from datetime import timedelta

def calculate_moving_average(df, period, mode, price_col):
    if mode == 0:  # SMA
        return df[price_col].rolling(window=period).mean()
    elif mode == 1:  # EMA
        return df[price_col].ewm(span=period, adjust=False).mean()
    elif mode == 2:  # SMMA (similar to WMA in some cases)
        sma = df[price_col].rolling(window=period).mean()
        smma = sma.ewm(span=period, adjust=False).mean()
        return smma
    elif mode == 3:  # LWMA
        weights = np.arange(1, period + 1)
        def lwma(prices):
            return np.dot(prices, weights) / weights.sum()
        return df[price_col].rolling(window=period).apply(lwma, raw=True)
    else:
        raise ValueError("Invalid mode. Use 0 for SMA, 1 for EMA, 2 for SMMA, 3 for LWMA.")

def calculate_cross_signals(df, faster_ma_period, faster_ma_mode, slower_ma_period, slower_ma_mode):
    df['FasterMA'] = calculate_moving_average(df, faster_ma_period, faster_ma_mode, 'close')
    df['FasterMA_prev'] = df['FasterMA'].shift(1)
    df['FasterMA_after'] = df['FasterMA'].shift(-1)
    
    df['SlowerMA'] = calculate_moving_average(df, slower_ma_period, slower_ma_mode, 'open')
    df['SlowerMA_prev'] = df['SlowerMA'].shift(1)
    df['SlowerMA_after'] = df['SlowerMA'].shift(-1)
    
    df['AvgRange'] = df[['high', 'low']].apply(lambda x: np.mean(np.abs(x['high'] - x['low'])), axis=1).rolling(window=10).mean()
    
    df['CrossUp'] = np.where((df['FasterMA'] > df['SlowerMA']) &
                             (df['FasterMA_prev'] < df['SlowerMA_prev']) &
                             (df['FasterMA_after'] > df['FasterMA_after']),
                             df['low'] - df['AvgRange'] * 0.3, np.nan)
    
    df['CrossDown'] = np.where((df['FasterMA'] < df['SlowerMA']) &
                               (df['FasterMA_prev'] > df['SlowerMA_prev']) &
                               (df['FasterMA_after'] < df['FasterMA_after']),
                               df['high'] + df['AvgRange'] * 0.3, np.nan)
    
    return df
# Sample usage
# data = {
#     'timestamp': pd.date_range(start='2023-01-01', periods=100, freq='D'),
#     'open': np.random.rand(100),
#     'high': np.random.rand(100),
#     'low': np.random.rand(100),
#     'close': np.random.rand(100),
#     'volume': np.random.rand(100)
# }
dataframe = pd.read_csv('/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/EURJPY_5_Min_testing_new.csv')
# df = pd.DataFrame(data)

faster_ma_period = 3
faster_ma_mode = 3
slower_ma_period = 3
slower_ma_mode = 3

final_df = calculate_cross_signals(dataframe, faster_ma_period, faster_ma_mode, slower_ma_period, slower_ma_mode)
final_df['UTC'] = pd.to_datetime(final_df['datetime']) + timedelta(hours=5)
final_df['GMT'] = final_df['UTC'] + timedelta(hours=2)
output_file = '/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/outputimmarrowsEurusd.csv'
final_df.to_csv(output_file, index=False)